# B2-023-generative-models-diffusion — Practice p21 — Solution

**Type:** scenario · **Difficulty:** core · **Concepts:** generative-adversarial-network, denoising-diffusion-probabilistic-models, stable-diffusion

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Solution

Speed is counted in network evaluations per generated sample: GAN $1$ (generator), VAE $1$ (decoder), DDPM $T$ (here $T=50$), and $2T=100$ for a DDPM with classifier-free guidance. Scoring an input under a VAE costs one encoder and one decoder pass for a single-sample ELBO estimate.

### (I) Real-time variations — **GAN**

**Justification.** The binding constraint is speed: thousands of samples per second on a phone CPU rules out a $T$-step sampler ($50$–$1000$ evaluations per sample), while a GAN has a **single-pass sampler** (one generator evaluation). Artists judge sharpness, which adversarial training targets directly (the discriminator penalizes blurry outputs that a VAE's Gaussian decoder tends to produce), and no likelihood is needed, so the GAN's implicit density costs nothing. Abundant data helps keep the discriminator from memorizing and mitigates instability.

**Falsifying held-out check.** If the p08 mode-coverage counts of fixed-noise samples against the layout clusters of held-out sprites leave clusters below the $\lceil n/(8K)\rceil$ threshold (mode collapse), or the mean discriminator score on held-out sprites differs from that on training sprites by more than $0.10$, the GAN recommendation fails.

### (II) Calibrated anomaly ranking — **VAE**

**Justification.** The team needs a score with a stated relation to $\log p(x)$, which the VAE's **ELBO** provides: $\mathrm{ELBO}(x)\le\log p_\theta(x)$, a lower bound that can rank readings. One encoder pass plus one decoder pass gives the score, so $10^5$ readings per second on 8-D inputs is realistic, whereas a GAN has no density at all and a diffusion bound needs $T$ network terms per reading. Sampling is rarely needed, so the VAE's blurrier samples do not matter.

**Falsifying held-out check.** If held-out normal readings receive a mean ELBO much lower than training readings (the bound is not calibrated beyond the training set), or held-out readings with known injected anomalies are not ranked below the held-out normal ones, the recommendation fails.

### (III) Rare modes matter — **diffusion model (DDPM)**

**Justification.** Coverage is the binding constraint and latency is not: a DDPM is trained with a simple regression loss (**stable training**, no adversary that can reward collapse) on every training row at every noise level, so it tends to **cover** all modes, including rare ones, as p19's run covered all four mixture components while p17's GAN covered two. The $T$ evaluations per sample are acceptable offline. A VAE would also cover modes but would smear the well-separated modes into each other.

**Falsifying held-out check.** If p08 coverage counts of a large fixed-noise sample show either rare mode below its threshold, or per-mode sample frequencies differ from the held-out per-mode frequencies by more than their sampling error, the recommendation fails.

### (IV) Text-conditioned images — **latent diffusion (Stable Diffusion style)**

**Justification.** Diffusing in a frozen autoencoder's **latent space** cuts each of the $2T$ denoiser evaluations to a $64\times64\times4$ tensor instead of $512\times512\times3$, which is what makes a single limited-memory GPU feasible. The prompt enters through **cross-attention** (latent queries, text-context keys and values), and **classifier-free guidance** with a weight $w$ gives exactly the adjustable prompt adherence requested, at the cost of two denoiser evaluations per step. GANs offer no comparably stable text conditioning at this scale, and a pixel-space VAE is too blurry.

**Falsifying held-out check.** On held-out prompts, if a fixed prompt-adherence measure (for example a held-out classifier's hit rate on the requested attribute, as in p20) does not increase from $w=0$ to $w=1$ to a moderate $w>1$, or the decoded latent round trip of held-out images is not faithful, the recommendation fails.

### (V) CPU prototype with a certificate — **DDPM (tiny 2-D, as in p18–p19)**

**Justification.** The p18/p19 DDPM trains in seconds on CPU on the four-component mixture and comes with the full set of certificates the project asks for: **coverage/statistics certificates** — p08 coverage counts per component, the inside fraction against starting noise, and mean and covariance gaps against held-out rows — plus a held-out denoising loss against a zero-predictor baseline. The GAN of p17 runs as fast but certifies only partial coverage, and a VAE cannot match the separated blobs as precisely.

**Falsifying held-out check.** If the p19 certificates fail on the 64 held-out rows — a coordinate mean gap above $0.35$, a covariance gap above $0.30$, fewer than $4$ covered components, or an inside fraction below $0.75$ — the recommendation fails.

In [ ]:
T = 50
evals_per_sample = {"GAN": 1, "VAE": 1, "DDPM": T, "DDPM+CFG": 2 * T}
decisions = {
    "I": "GAN",
    "II": "VAE",
    "III": "DDPM",
    "IV": "latent diffusion",
    "V": "DDPM",
}
# Case I needs thousands of samples per second on a phone CPU: compare evaluation budgets.
budget_ratio_I = evals_per_sample["DDPM"] / evals_per_sample["GAN"]
# Case IV: guided latent diffusion per step costs 2 evaluations, each on a 48x smaller tensor than pixels.
latent_size, pixel_size = 64 * 64 * 4, 512 * 512 * 3
print(evals_per_sample, decisions, budget_ratio_I, pixel_size / latent_size)

### Answer check

In [ ]:
assert evals_per_sample == {"GAN": 1, "VAE": 1, "DDPM": 50, "DDPM+CFG": 100}
assert decisions == {"I": "GAN", "II": "VAE", "III": "DDPM", "IV": "latent diffusion", "V": "DDPM"}
assert budget_ratio_I == 50
assert pixel_size / latent_size == 48